# PI Controller Tuner

Interactive PI tuning tool — equivalent to the MATLAB PID Tuner.

All analysis functions live in **`pi_tuner_lib.py`** (same folder).  
**Only Cell 1 needs to be edited** for a normal tuning session.

---

## Transfer Function

The PI controller used here:

$$C(s) = K_p + \frac{K_i}{s} = \frac{K_p s + K_i}{s}$$

Where:
- $K_p$ — Proportional gain
- $K_i$ — Integral gain

The open-loop and closed-loop systems are:

$$G_{OL}(s) = C(s) \cdot G(s) \qquad G_{CL}(s) = \frac{G_{OL}(s)}{1 + H(s)\,G_{OL}(s)}$$

Where $G(s)$ is the **plant** and $H(s)$ is the **feedback/sensor** transfer function ($H = 1$ for unity feedback).

---

## 📚 Stability Reminder

| Indicator | Target | Meaning |
|-----------|--------|---------|
| **Phase Margin** | > 45° | Distance to instability on the phase axis |
| **Gain Margin** | > 6 dB | Distance to instability on the gain axis |
| **Kp ↑** | Faster response | May reduce margins → risk of instability |
| **Ki ↑** | Eliminates SS error | Adds phase lag → reduces phase margin |


## 1 — Imports

In [1]:
import numpy as np
import control as ct

from src.pi_tuner_lib import (
    s,                    # Laplace variable  — use to write transfer functions
    build_pi,
    compute_bode,
    compute_margins,
    compute_step_response,
    get_poles_zeros,
    create_pi_tuner,
    print_analysis,
)

print("Imports OK — library loaded from pi_tuner_lib.py")


Imports OK — library loaded from pi_tuner_lib.py


## 2 — Configuration  ← edit this cell

In [2]:
# =============================================================================
#  USER CONFIGURATION  
# =============================================================================

# --- INITIAL GAINS -----------------------------------------------------------
Kp_INIT = 1.0        # Initial proportional gain
Ki_INIT = 10.0       # Initial integral gain

# --- SLIDER RANGES  (log scale) ----------------------------------------------
Kp_RANGE = (1e-4, 1e4)   # (min, max) for Kp slider
Ki_RANGE = (1e-4, 1e5)   # (min, max) for Ki slider

# --- BODE PLOT SETTINGS ------------------------------------------------------
F_MIN  = 0.1         # Hz — lower frequency bound
F_MAX  = 10_000.0    # Hz — upper frequency bound
N_FREQ = 2000        # number of frequency points (more = sharper plot, slower)

# --- STEP RESPONSE SETTINGS --------------------------------------------------
T_END  = 1.0e-2       # s  — simulation end time
N_TIME = 10000        # number of time samples

# --- FIGURE HEIGHT -----------------------------------------------------------
FIG_HEIGHT = 440     # px — height of each Bode plot panel

# =============================================================================
print("Configuration loaded.")
print(f"  Kp init  : {Kp_INIT}   Ki init : {Ki_INIT}")
print(f"  Frequency: {F_MIN} Hz → {F_MAX} Hz")
print(f"  Step time: 0 → {T_END} s")


Configuration loaded.
  Kp init  : 1.0   Ki init : 10.0
  Frequency: 0.1 Hz → 10000.0 Hz
  Step time: 0 → 0.01 s


## 3 — Plant & Feedback Transfer Functions

Define `plant_tf` and (optionally) `feedback_tf` using the `control` library.  
The variable `s` is already imported as `ct.tf('s')` — use it for natural notation.

> **Tip**: `ct.tf(num, den)` also accepts coefficient lists, e.g.  
> `ct.tf([1], [1e-3, 0.01])` → 1 / (0.001s + 0.01)


In [3]:
# =============================================================================
#  PLANT DEFINITION  — edit to match your system
# =============================================================================

# --- Example A: Simple RL filter  G(s) = 1 / (L·s + R) ----------------------
L   = 25e-3      # Inductance  [H]
R_L = 25       # Resistance  [Ω]
Ue = 140           # Input voltage [V]
f_sw = 16e3        # Switching frequency [Hz]


G_sys = 1 / (L*s + R_L)

Gcm = Ue / (1+(s/(3*f_sw)))

plant_tf = G_sys*Gcm

# --- Example B: Second-order system ------------------------------------------
# wn = 2 * np.pi * 50        # Natural frequency [rad/s]
# zeta = 0.3                 # Damping ratio
# plant_tf = wn**2 / (s**2 + 2*zeta*wn*s + wn**2)

# --- Example C: Integrating plant --------------------------------------------
# K_int = 5.0
# plant_tf = K_int / s

# --- Example D: Transfer function from num/den lists -------------------------
# plant_tf = ct.tf([1], [1e-3, 0.01, 0])   # 1 / (0.001s² + 0.01s)

# --- Example E: With time delay (Padé approximation) -------------------------
# T_d = 500e-6               # Dead-time [s]
# delay = ct.tf([-T_d/2, 1], [T_d/2, 1])   # First-order Padé
# plant_tf = delay / (L*s + R_L)


# =============================================================================
#  FEEDBACK / SENSOR TRANSFER FUNCTION  — set to None for unity feedback
# =============================================================================

feedback_tf = None          # Unity feedback  H(s) = 1

# --- Example: Low-pass sensor  H(s) = 1 / (τ·s + 1) -------------------------
# tau_sensor = 1e-4
# feedback_tf = 1 / (tau_sensor*s + 1)


# =============================================================================
print("Plant TF:")
print(plant_tf)
if feedback_tf is not None:
    print("\nFeedback TF:")
    print(feedback_tf)
else:
    print("\nFeedback TF: unity (H = 1)")


Plant TF:
<TransferFunction>: sys[12]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']

            6.72e+06
  ----------------------------
  0.025 s^2 + 1225 s + 1.2e+06

Feedback TF: unity (H = 1)


## 4 — Interactive PI Tuner

Run this cell to launch the tuner.  
Use the **Kp** and **Ki** sliders to tune interactively — all four plots update in real time:

| Plot | Content |
|------|---------|
| **Open-Loop Bode** | Magnitude + phase with Phase Margin and Gain Margin markers |
| **Closed-Loop Bode** | Magnitude + phase of the closed-loop system |
| **Step Response** | Unit step response with rise time, settling time, overshoot |
| **Pole-Zero Map** | Open-loop and closed-loop poles (×) and zeros (○) |


In [4]:
create_pi_tuner(
    plant_tf    = plant_tf,
    feedback_tf = feedback_tf,
    init_Kp     = Kp_INIT,
    init_Ki     = Ki_INIT,
    f_min       = F_MIN,
    f_max       = F_MAX,
    n_freq      = N_FREQ,
    t_end       = T_END,
    n_time      = N_TIME,
    Kp_range    = Kp_RANGE,
    Ki_range    = Ki_RANGE,
    fig_height  = FIG_HEIGHT,
)


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

...

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

## 5 — Detailed Analysis Report

After finding good gains with the interactive tuner, enter them below to print
a precise numerical report (margins, crossover frequencies, pole locations, 
step metrics).


In [5]:
# =============================================================================
#  ANALYSIS AT SPECIFIC GAINS  — paste your tuned values here
# =============================================================================

Kp_REPORT = Kp_INIT   # ← replace with your tuned Kp
Ki_REPORT = Ki_INIT   # ← replace with your tuned Ki

print_analysis(
    plant_tf    = plant_tf,
    feedback_tf = feedback_tf,
    Kp          = Kp_REPORT,
    Ki          = Ki_REPORT,
    t_end       = T_END,
    n_time      = N_TIME,
)


──────────────────────────────────────────────────────
  PI CONTROLLER — ANALYSIS REPORT
──────────────────────────────────────────────────────
  Kp = 1   |   Ki = 10
  C(s) = 1 + 10 / s

  ── Open-Loop Stability ─────────────────────────────
  ✓ STABLE
  Phase margin        : 93.74 °
  Gain margin         : ∞
  Gain crossover freq : 871.11 Hz   (|G_OL| = 1)
  Phase crossover freq: —   (∠G_OL = −180°)

  ── Closed-Loop Step Response ───────────────────────
  Rise time  (10–90 %) : 0.31 ms
  Settling time  (2 %) : 0.68 ms
  Overshoot            : 0.00 %
  Steady-state gain    : 0.8617

  ── Closed-Loop Poles ───────────────────────────────
       -8.4960      +0.0000j   ζ = 1.000   ωn = 8.496 rad/s
    -7653.5996      +0.0000j   ζ = 1.000   ωn = 7654 rad/s
   -41337.9044      +0.0000j   ζ = 1.000   ωn = 4.134e+04 rad/s
──────────────────────────────────────────────────────


## 6 — Custom Open-Loop Bode  *(re-run independently)*

Plot the open-loop response for any gain combination without re-running the full tuner.  
Edit **only this cell** and re-run it independently.


In [6]:
# =============================================================================
#  CUSTOM OPEN-LOOP BODE  ← edit freely and re-run this cell alone
# =============================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.pi_tuner_lib import build_pi, compute_bode, compute_margins

# Gains to compare
COMPARE_GAINS = [
    (Kp_INIT,  Ki_INIT,  '#2196F3', 'Initial'),
    #(5.0,  50.0,  '#FF5722', 'Tuned'),   # ← add more rows to overlay
]

freqs = np.geomspace(F_MIN, F_MAX, N_FREQ)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=['Magnitude', 'Phase'])

for (Kp_c, Ki_c, color, label) in COMPARE_GAINS:
    C_c    = build_pi(Kp_c, Ki_c)
    G_OL_c = C_c * plant_tf
    m_c, p_c = compute_bode(G_OL_c, freqs)

    fig.add_trace(go.Scatter(x=freqs, y=m_c, mode='lines',
                             name=f'{label}  Kp={Kp_c:.3g}, Ki={Ki_c:.3g}',
                             line=dict(color=color, width=2.0)), row=1, col=1)
    fig.add_trace(go.Scatter(x=freqs, y=p_c, mode='lines',
                             name=f'{label}  Kp={Kp_c:.3g}, Ki={Ki_c:.3g}',
                             line=dict(color=color, width=2.0),
                             showlegend=False), row=2, col=1)

fig.add_hline(y=0,    line_dash='dash', line_color='#aaaaaa', line_width=1.0, row=1, col=1)
fig.add_hline(y=-180, line_dash='dash', line_color='#aaaaaa', line_width=1.0, row=2, col=1)

fig.update_xaxes(type='log', showgrid=True, gridcolor='#ebebeb',
                 title_text='Frequency (Hz)', row=2, col=1)
fig.update_xaxes(type='log', showgrid=True, gridcolor='#ebebeb', row=1, col=1)
fig.update_yaxes(title_text='Magnitude (dB)', showgrid=True,
                 gridcolor='#ebebeb', row=1, col=1)
fig.update_yaxes(title_text='Phase (°)', showgrid=True,
                 gridcolor='#ebebeb', row=2, col=1)
fig.update_layout(
    height=500, title_text='Open-Loop Bode — Gain Comparison',
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=65, r=15, t=60, b=15),
    legend=dict(orientation='h', y=-0.10, x=0),
    hovermode='x unified',
)
fig.show()
